## Reporting variability honestly

Report every seed, mean, standard deviation, range, and the baseline. With only a few seeds, do not imply a precisely estimated probability distribution. For data sufficiency, show error versus number and location of development Reynolds cases; count alone can hide poor coverage.

To evaluate an uncertainty indicator, compare spread and actual error across complete blind cases. Ranking agreement is a useful first check, but calibration requires coverage analysis at declared intervals or thresholds.


# Project Track 4 - Uncertainty and Data-Sufficiency Study

**Choose one variant:**

- **4A Training-seed uncertainty:** repeat the same DNN training and build an ensemble uncertainty map.
- **4B Training-case sufficiency:** hold architecture fixed and vary only the number of Reynolds-number cases.

The goal is to distinguish one lucky training run from a reproducible scientific result.

## Required files
`P4_Uncertainty_Study.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`

<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## Learning objectives: uncertainty is not one number

This track studies two different sources of variability. Variant 4A changes training initialization while holding data fixed. Variant 4B changes the amount of development information while holding the test and training protocol fixed. You should be able to report distributions rather than a preferred run and test whether a proposed uncertainty indicator tracks actual blind error.

Prerequisites: random seeds, case-wise splits, ensemble mean/spread, validation-only model selection, and confidence versus calibration.


In [ ]:
# Repository/Colab bootstrap. Run this before the import cell below.
from pathlib import Path
import sys

def _find_course_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "common" / "w5_common.py").exists():
            return base
    # Colab flat-upload fallback: helper files and data beside the notebook.
    if (start / "w5_common.py").exists():
        return start
    raise FileNotFoundError(
        "Course root not found. Clone the repository, or upload w4utils.py, "
        "w5_common.py, the track-specific helpers, and cavity_data.npz as listed above."
    )

COURSE_ROOT = _find_course_root()
COMMON_DIR = COURSE_ROOT / "common" if (COURSE_ROOT / "common").exists() else COURSE_ROOT
DATASET_PATH = COURSE_ROOT / "data" / "cavity_data.npz"
if not DATASET_PATH.exists():
    DATASET_PATH = COURSE_ROOT / "cavity_data.npz"
sys.path.insert(0, str(COMMON_DIR))
print("Course root:", COURSE_ROOT)
print("Common helpers:", COMMON_DIR)
print("Dataset:", DATASET_PATH)


In [ ]:
import time,importlib
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import w4utils,w5_common
importlib.reload(w4utils);importlib.reload(w5_common)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data=w5_common.require_week4_files(str(DATASET_PATH)); VARIANT="4A"
VAL_RE=300; TEST_RE=[175,275,375]; HIDDEN=(64,64,64)


## A practical uncertainty taxonomy

- **Numerical error:** discretization, solver convergence, pressure recovery, or finite sampling window in the labels.
- **Aleatoric/statistical variability:** irreducible sampling variation in stochastic particle data.
- **Epistemic/model variability:** sensitivity to limited data, architecture, initialization, or training.
- **Distribution shift:** blind physical conditions that are not represented by development support.

An ensemble of training seeds probes only a narrow part of epistemic variability. Small ensemble spread does not prove low error; all members can be confidently wrong under extrapolation.


## 4A. Repeated training seeds

Architecture, data, optimizer, and stopping rule remain fixed. Only initialization and stochastic optimization change.

In [ ]:
SEEDS=[11,22,33,44,55]
BASE_TRAIN=[100,150,200,225,250,350,400]
seed_rows=[];seed_predictions={}
if VARIANT=="4A":
    for seed in SEEDS:
        t0=time.time()
        b=w5_common.train_pointwise_model(data,BASE_TRAIN,VAL_RE,hidden=HIDDEN,
            stride=2,seed=seed,epochs=850,patience=60)
        seed_predictions[seed]={}
        for r in TEST_RE:
            pred=w5_common.predict_case(b,r,data["x"],data["y"])
            seed_predictions[seed][r]=pred
            seed_rows.append({"seed":seed,"Re":r,"training_seconds":time.time()-t0,
                              **w5_common.evaluate_prediction(data,r,pred)})
    seed_results=pd.DataFrame(seed_rows);display(seed_results)


## Design the comparison before running it

For a seed study, keep case lists, architecture, epoch budget, scaling rule, and metrics identical. For a data-sufficiency study, use nested development sets so that “more data” really adds cases rather than replacing them. Do not change two factors and attribute the outcome to one.

**Prediction prompt:** Will ensemble spread be largest where the true error is largest? Write a reason it might succeed and a reason it might fail.


## 4B. Number of development cases

The architecture and spatial sampling stay fixed. Only the number and coverage of physical cases change. The labels below count **development cases**, including the fixed validation case at `Re=300`; the result table reports both development-case and actual training-case counts.

In [ ]:
DEV_CASE_SETS={
    "4_dev_cases":[100,200,300,400],
    "6_dev_cases":[100,150,200,250,300,400],
    "8_dev_cases":[100,150,200,225,250,300,350,400],
}
size_rows=[];size_predictions={}
if VARIANT=="4B":
    for label,cases in DEV_CASE_SETS.items():
        val=300; train=[r for r in cases if r!=val]
        t0=time.time()
        b=w5_common.train_pointwise_model(data,train,val,hidden=HIDDEN,
            stride=2,seed=690,epochs=850,patience=60)
        size_predictions[label]={}
        for r in TEST_RE:
            pred=w5_common.predict_case(b,r,data["x"],data["y"])
            size_predictions[label][r]=pred
            size_rows.append({"case_set":label,"n_dev_cases":len(cases),
                              "n_train_cases":len(train),"Re":r,
                              "training_seconds":time.time()-t0,
                              **w5_common.evaluate_prediction(data,r,pred)})
    size_results=pd.DataFrame(size_rows);display(size_results)


## 3. Ensemble or data-sufficiency evidence

In [ ]:
if VARIANT=="4A":
    r=275; stack_u=np.stack([seed_predictions[s][r][0] for s in SEEDS]);
    stack_v=np.stack([seed_predictions[s][r][1] for s in SEEDS]);
    stack_p=np.stack([seed_predictions[s][r][2] for s in SEEDS]);
    ensemble=(stack_u.mean(0),stack_v.mean(0),stack_p.mean(0))
    idx=int(np.where(data["Re"]==r)[0][0])
    spread=np.sqrt(stack_u.std(0)**2+stack_v.std(0)**2)
    error=np.hypot(ensemble[0]-data["u"][idx],ensemble[1]-data["v"][idx])
    corr=np.corrcoef(spread.ravel(),error.ravel())[0,1]
    print("correlation between ensemble spread and actual vector error:",corr)
    X,Y=np.meshgrid(data["x"],data["y"])
    fig,ax=plt.subplots(1,3,figsize=(13,3.7))
    a=ax[0].contourf(X,Y,error,28);fig.colorbar(a,ax=ax[0]);ax[0].set_title("ensemble-mean error")
    a=ax[1].contourf(X,Y,spread,28);fig.colorbar(a,ax=ax[1]);ax[1].set_title("training-seed spread")
    ax[2].scatter(spread.ravel(),error.ravel(),s=5,alpha=.4);ax[2].set(xlabel="spread",ylabel="error",title=f"correlation={corr:.2f}")
    plt.tight_layout();plt.show()
    seed_results.to_csv("P4_seed_results.csv",index=False)
else:
    size_results.to_csv("P4_data_size_results.csv",index=False)
    fig,ax=plt.subplots(1,2,figsize=(10,3.8))
    for label,g in size_results.groupby("case_set"):
        ax[0].plot(g["Re"],g["relative_L2_uv"],"o-",label=label)
        ax[1].plot(g["Re"],g["relative_L2_p"],"o-",label=label)
    for a in ax:a.grid(.3);a.legend();a.set_xlabel("blind Re")
    ax[0].set_ylabel("relative L2 velocity");ax[1].set_ylabel("relative L2 pressure")
    plt.tight_layout();plt.show()


## Required report evidence

### Variant 4A
- Report mean, standard deviation, best, and worst error across seeds.
- Compare one model with the ensemble mean.
- Plot uncertainty/spread and actual error on the same blind case.
- Explain whether ensemble spread is a useful error indicator and where it fails.

### Variant 4B
- Keep network and training rules fixed.
- Report error versus both the number of development cases and the actual number of training cases, not merely the number of flattened grid points.
- Explain whether added cases improve interpolation, extrapolation, or both.
- Identify the smallest case set that preserves the central flow but loses a wall/corner feature.

For both variants, a result without repeated runs or a controlled data change is not an uncertainty study.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Evaluate whether ensemble spread is calibrated by plotting empirical error coverage at several spread thresholds.
2. Combine training-seed variability with development-case count in a small two-factor experiment.
3. Use ensemble disagreement as an acquisition score and test one simple active-learning choice against a random added Reynolds case.


## Concept check and further reading

1. What uncertainty source is measured by changing only neural initialization?
2. Why can five ensemble members agree and still be wrong?
3. Why should development sets be nested in a data-density study?
4. How would label uncertainty alter the interpretation of a seed ensemble?
5. Which additional experiment would separate numerical-label error from model error?

Read the reproducibility references in `references/README.md` and the uncertainty discussion in Karniadakis et al. (2021).


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
